# Traffic Demand Prediction

This notebook is my attempt at the Traffic Demand Prediction hackathon on HackerEarth.

The goal is to predict traffic demand at various geo-locations given road/weather info.

Evaluation metric: `max(0, 100 * R2_score(actual, predicted))`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## Load the Data

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample = pd.read_csv('sample_submission.csv')

print("Train:", train.shape)
print("Test: ", test.shape)
train.head()

## Quick EDA

In [ ]:
train.info()

In [ ]:
# check nulls
print("Train nulls:")
print(train.isnull().sum())
print("\nTest nulls:")
print(test.isnull().sum())

In [ ]:
train['demand'].describe()

In [ ]:
# demand distribution - quite skewed
plt.figure(figsize=(8,4))
train['demand'].hist(bins=50)
plt.title('Demand Distribution')
plt.xlabel('demand')
plt.show()

In [ ]:
# check categorical columns
print("Weather:", train['Weather'].value_counts().to_dict())
print("RoadType:", train['RoadType'].value_counts().to_dict())
print("LargeVehicles:", train['LargeVehicles'].value_counts().to_dict())
print("Landmarks:", train['Landmarks'].value_counts().to_dict())

## Feature Engineering

Timestamps are in H:M format so I'll extract hour and minute separately.
Also adding cyclical features for time since 23:59 should be close to 0:00.


In [ ]:
def feature_engineer(df):
    df = df.copy()
    
    # parse timestamp
    df['hour'] = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['minute'] = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df['time_of_day'] = df['hour'] + df['minute'] / 60.0
    
    # cyclical time features - sin/cos so model knows 23:00 is close to 0:00
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['time_sin'] = np.sin(2 * np.pi * df['time_of_day'] / 24)
    df['time_cos'] = np.cos(2 * np.pi * df['time_of_day'] / 24)
    
    # encode categoricals
    df['RoadType_enc'] = df['RoadType'].map({'Residential': 0, 'Street': 1, 'Highway': 2}).fillna(-1)
    df['LargeVehicles_enc'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['Landmarks_enc'] = (df['Landmarks'] == 'Yes').astype(int)
    df['Weather_enc'] = df['Weather'].map({'Sunny': 0, 'Foggy': 1, 'Rainy': 2, 'Snowy': 3}).fillna(-1)
    
    # geohash - just label encode for now
    df['geohash_enc'] = pd.Categorical(df['geohash']).codes
    
    # fill missing temperature with median
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())
    
    # some interaction features
    df['road_lanes'] = df['RoadType_enc'] * df['NumberofLanes']
    df['is_rush_hour'] = (((df['hour'] >= 7) & (df['hour'] <= 9)) | 
                          ((df['hour'] >= 17) & (df['hour'] <= 19))).astype(int)
    df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
    
    return df

train_fe = feature_engineer(train)
test_fe = feature_engineer(test)

print("Done!")
train_fe.head()

In [ ]:
FEATURES = [
    'day', 'hour', 'minute', 'time_of_day',
    'hour_sin', 'hour_cos', 'time_sin', 'time_cos',
    'RoadType_enc', 'NumberofLanes', 'LargeVehicles_enc',
    'Landmarks_enc', 'Temperature', 'Weather_enc',
    'geohash_enc', 'road_lanes', 'is_rush_hour', 'is_night'
]

X = train_fe[FEATURES]
y = train_fe['demand']
X_test = test_fe[FEATURES]

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)

## Model - LightGBM

Tried a few models, LightGBM gave the best CV scores.


In [ ]:
model = LGBMRegressor(
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=128,
    max_depth=8,
    min_child_samples=20,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

In [ ]:
# 5-fold CV to check score before submitting
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)
print("CV R2 scores:", cv_scores.round(4))
print("Mean:", round(cv_scores.mean(), 4))
print("Hackathon score estimate:", round(max(0, 100 * cv_scores.mean()), 2))

In [ ]:
# train on full data
model.fit(X, y)
print("Training done")

## Feature Importance

In [ ]:
feat_imp = pd.DataFrame({
    'feature': FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feat_imp)
plt.title('Feature Importances')
plt.tight_layout()
plt.show()

## Predict & Save Submission

In [ ]:
preds = model.predict(X_test)
# clip to [0,1] since demand is bounded
preds = np.clip(preds, 0, 1)

submission = pd.DataFrame({'Index': test['Index'], 'demand': preds})
submission.to_csv('submission.csv', index=False)

print("Saved submission.csv")
print("Shape:", submission.shape)
submission.head(10)

In [ ]:
# quick sanity check
print("Min:", submission['demand'].min())
print("Max:", submission['demand'].max())
print("Mean:", submission['demand'].mean().round(4))
print("Nulls:", submission['demand'].isnull().sum())